In [1]:
import sys
print(sys.version)

3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]


In [ ]:
# ============================================================
# TASK 2 - GOOGLE PLAY STORE ANALYTICS
# COMPLETE IMPLEMENTATION IN ONE CELL
# ============================================================

import os
import webbrowser
import pandas as pd
import numpy as np
import plotly.express as px

from datetime import datetime, time
from zoneinfo import ZoneInfo

# ============================================================
# 1. LOAD DATASET
# ============================================================

file_path = "Play Store Data.csv"

apps_df = pd.read_csv(file_path)

print("✅ Dataset loaded successfully")
print("Rows:", apps_df.shape[0])
print("Columns:", apps_df.shape[1])

# ============================================================
# 2. CLEAN INSTALLS COLUMN
# ============================================================

task2_df = apps_df.copy()

task2_df["Installs"] = (
    task2_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

task2_df["Installs"] = pd.to_numeric(
    task2_df["Installs"],
    errors="coerce"
)

task2_df = task2_df.dropna(
    subset=["Installs", "Category"]
).copy()

task2_df["Installs"] = task2_df["Installs"].astype(int)

print("✅ Installs cleaned successfully")

# ============================================================
# 3. EXCLUDE A, C, G AND S CATEGORIES
# ============================================================

excluded_letters = ("A", "C", "G", "S")

task2_df = task2_df[
    ~task2_df["Category"]
    .astype(str)
    .str.upper()
    .str.startswith(excluded_letters)
].copy()

print("✅ A, C, G and S categories excluded")

# ============================================================
# 4. FIND TOP 5 CATEGORIES
# ============================================================

top5 = (
    task2_df
    .groupby("Category")["Installs"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

top5_categories = top5.index.tolist()

print("✅ Top 5 eligible categories:")
print(top5_categories)

# ============================================================
# 5. COUNTRY LIST FOR CHOROPLETH
# ============================================================

countries = [
    "India",
    "United States",
    "Canada",
    "Brazil",
    "Mexico",
    "United Kingdom",
    "France",
    "Germany",
    "Italy",
    "Spain",
    "Australia",
    "Japan",
    "South Korea",
    "Indonesia",
    "Saudi Arabia",
    "South Africa",
    "Nigeria",
    "Turkey",
    "Russia",
    "Argentina"
]

# ============================================================
# 6. KEEP ONLY TOP 5 CATEGORIES
# ============================================================

task2_top5 = task2_df[
    task2_df["Category"].isin(top5_categories)
].copy()

# ============================================================
# 7. PREPARE CHOROPLETH DATA
# ============================================================

category_totals = (
    task2_top5
    .groupby("Category", as_index=False)["Installs"]
    .sum()
    .rename(columns={"Installs": "Global_Installs"})
)

map_rows = []

for _, row in category_totals.iterrows():

    for country in countries:

        map_rows.append({
            "Country": country,
            "Category": row["Category"],
            "Global_Installs": row["Global_Installs"]
        })

map_df = pd.DataFrame(map_rows)

# Highlight categories above 1 million installs
map_df["Install_Status"] = np.where(
    map_df["Global_Installs"] > 1_000_000,
    "Above 1 Million",
    "1 Million or Less"
)

print("✅ Choropleth map data prepared")

# ============================================================
# 8. CHECK CURRENT IST TIME
# ============================================================

ist = ZoneInfo("Asia/Kolkata")

now_ist = datetime.now(ist)

current_time = now_ist.time()

start_time = time(18, 0)   # 6:00 PM
end_time = time(20, 0)     # 8:00 PM

print(
    "Current IST time:",
    now_ist.strftime("%I:%M:%S %p")
)

# ============================================================
# 9. DASHBOARD PATH
# ============================================================

dashboard_path = os.path.abspath(
    "dashboard_task2.html"
)

# ============================================================
# 10. TASK 2 ACTIVE: 6 PM - 8 PM IST
# ============================================================

if start_time <= current_time < end_time:

    print("✅ TASK 2 IS ACTIVE")
    print("✅ Current time is between 6 PM and 8 PM IST")

    # --------------------------------------------------------
    # CREATE INTERACTIVE CHOROPLETH
    # --------------------------------------------------------

    fig_task2 = px.choropleth(
        map_df,
        locations="Country",
        locationmode="country names",
        color="Global_Installs",
        animation_frame="Category",
        hover_name="Country",
        hover_data={
            "Category": True,
            "Global_Installs": ":,",
            "Install_Status": True
        },
        color_continuous_scale="Viridis",
        title="Global Installs by Top 5 App Categories"
    )

    fig_task2.update_geos(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth"
    )

    fig_task2.update_layout(
        height=650,
        margin=dict(
            l=20,
            r=20,
            t=80,
            b=20
        )
    )

    # --------------------------------------------------------
    # CREATE FINAL DASHBOARD
    # --------------------------------------------------------

    dashboard_html = f"""
    <!DOCTYPE html>

    <html>

    <head>

        <meta charset="UTF-8">

        <title>
            Task 2 - Google Play Store Analytics
        </title>

        <script src="https://cdn.plot.ly/plotly-latest.min.js">
        </script>

        <style>

            body {{
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 20px;
                background: #f5f5f5;
            }}

            .dashboard {{
                max-width: 1200px;
                margin: auto;
                background: white;
                padding: 25px;
                border-radius: 10px;
            }}

            h1, h2 {{
                text-align: center;
            }}

            .info {{
                padding: 15px;
                margin-bottom: 20px;
                background: #eeeeee;
                border-radius: 8px;
            }}

        </style>

    </head>

    <body>

        <div class="dashboard">

            <h1>
                Google Play Store Analytics
            </h1>

            <div class="info">

                <h2>
                    Task 2 - Global Installs by Top 5 App Categories
                </h2>

                <p>
                    Interactive Choropleth map showing global
                    installs for the Top 5 eligible app categories.
                </p>

                <p>
                    Categories beginning with A, C, G and S
                    are excluded.
                </p>

                <p>
                    Categories with installs above 1 million
                    are highlighted.
                </p>

                <p>
                    <strong>
                        Available Time:
                    </strong>
                    6:00 PM - 8:00 PM IST
                </p>

            </div>

            {fig_task2.to_html(
                full_html=False,
                include_plotlyjs=False
            )}

        </div>

    </body>

    </html>
    """

    # --------------------------------------------------------
    # SAVE FINAL DASHBOARD
    # --------------------------------------------------------

    with open(
        dashboard_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(dashboard_html)

    print("✅ Task 2 dashboard created")
    print("📁 File:", dashboard_path)

    # --------------------------------------------------------
    # OPEN ONLY ONE BROWSER OUTPUT
    # --------------------------------------------------------

    webbrowser.open(
        "file://" + dashboard_path
    )

    print("🌍 Interactive choropleth opened in browser")

# ============================================================
# 11. TASK 2 HIDDEN OUTSIDE 6 PM - 8 PM
# ============================================================

else:

    print("⏰ TASK 2 IS HIDDEN")

    print(
        "Task 2 is displayed only between "
        "6 PM and 8 PM IST."
    )

    # Remove previous dashboard if it exists
    if os.path.exists(dashboard_path):

        os.remove(dashboard_path)

        print(
            "🗑️ Previous Task 2 dashboard removed."
        )

    # --------------------------------------------------------
    # CREATE UNAVAILABLE PAGE
    # --------------------------------------------------------

    hidden_html = """
    <!DOCTYPE html>

    <html>

    <head>

        <meta charset="UTF-8">

        <title>
            Task 2 - Unavailable
        </title>

        <style>

            body {
                font-family: Arial, sans-serif;
                text-align: center;
                padding-top: 150px;
            }

        </style>

    </head>

    <body>

        <h1>
            ⏰ Task 2 is currently unavailable
        </h1>

        <h2>
            Task 2 is displayed only between
            6 PM and 8 PM IST.
        </h2>

        <p>
            Please check again during the allowed time.
        </p>

    </body>

    </html>
    """

    with open(
        dashboard_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(hidden_html)

    print(
        "📁 Unavailable page created:",
        dashboard_path
    )

    # --------------------------------------------------------
    # OPEN ONLY ONE RESTRICTION PAGE
    # --------------------------------------------------------

    webbrowser.open(
        "file://" + dashboard_path
    )

    print(
        "🌐 Task 2 restriction page opened in browser"
    )

✅ Dataset loaded successfully
Rows: 10841
Columns: 13
✅ Installs cleaned successfully
✅ A, C, G and S categories excluded
✅ Top 5 eligible categories:
['PRODUCTIVITY', 'TOOLS', 'FAMILY', 'PHOTOGRAPHY', 'NEWS_AND_MAGAZINES']
✅ Choropleth map data prepared
Current IST time: 06:29:23 PM
✅ TASK 2 IS ACTIVE
✅ Current time is between 6 PM and 8 PM IST
✅ Task 2 dashboard created
📁 File: c:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task2.html
🌍 Interactive choropleth opened in browser
